<a href="https://colab.research.google.com/github/areebaeman234-ux/ML-Internship/blob/main/Copy_of_w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window


**One row = one page per day**
- Each row represents a single page's performance on a single day

**Time window:**
- Development: March 2026 (mid-panel month)
- Testing (sealed): June 2026 (final month)

**Why this window:**
- March 2026 has complete data for feature development
- June 2026 is held back for honest testing

## 2. Fields: feature / label / context / excluded

Based on the data loaded above, here are the fields I identified:

### Features (what I'll use to predict):
| Field | Type | Description | Why it's knowable |
|-------|------|-------------|-------------------|
| gsc_avg_position | Numeric | Average ranking position in search results | Historical position data available |
| gsc_impressions | Numeric | How many times page appeared in search | Historical data available |
| gsc_clicks | Numeric | How many clicks page received | Historical data available |
| content_age_days | Numeric | Days since content was published | Publish date known |
| client_has_gsc | Boolean | Whether GSC data is available | Known metadata |

### Label (what I'm predicting):
| Field | Type | Description |
|-------|------|-------------|
| gsc_ctr | Numeric | Click-Through Rate = clicks / impressions |
| (or ga4_sessions) | Numeric | Website sessions as performance measure |

### Context (not used for prediction):
| Field | Type | Description |
|-------|------|-------------|
| content_hash_id | ID | Anonymized page identifier |
| client_hash_id | ID | Anonymized client identifier |
| report_date | Date | Date of observation |
| month | Text | Month of data |

### Excluded (why I'm not using these):
| Field | Why Excluded |
|-------|--------------|
| client_hash_id | Just an identifier, not a signal |
| content_hash_id | Just an identifier, not a signal |
| ga4_* columns (if client_has_ga4 = False) | Missing data - can't use |
| ai_* columns | Not all clients have this data |
| rows with gsc_impressions = 0 | No data to learn from |

### Handling Missing Data:
- For clients with `client_has_ga4 = False`: GA4 columns remain as NA (not 0)
- For clients with `client_has_ga4 = True`: Fill missing numeric values with 0
- Boolean columns: Fill NA with False

In [ ]:
# Install and load data
!pip install duckdb pyarrow
import duckdb
import pandas as pd
from google.colab import userdata

# Get token
hf_token = userdata.get('hf_token')

# Connect and load data
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

# Load sample of March 2026 data
query = """
    SELECT *
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
    WHERE month = '2026-03'
    LIMIT 100
"""
df = con.execute(query).df()

print("="*60)
print("📊 DATA LOADED SUCCESSFULLY")
print("="*60)
print(f"Rows: {len(df)}")
print(f"Columns: {len(df.columns)}")
print("\n📋 Column names:")
print(df.columns.tolist())
print("\n📊 First 5 rows:")
print(df.head())
print("\n📊 Data types:")
print(df.dtypes)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

📊 DATA LOADED SUCCESSFULLY
Rows: 100
Columns: 31

📋 Column names:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']

📊 First 5 rows:
  report_date           client_hash_id           content_hash_id  \
0  2026-03-01  client_73cda7b4e4f265ea  content_b7e512995f79d5a6   
1  2026-03-01  client_73cda7b4e4f265ea  content_05597932fe4da067   
2  2026-03-01  client_73cda7b4e4f265ea  content_7a105f548d9c6916   
3  2026-03-01  client_73cda7b4e4f265ea  content_905aa32a0230694e   
4  2026-03-01  client_73cda7b4e4f265ea  co

## 3. Verify it with queries (grain, counts, missing values, windows)
### Observations from Verification Queries:

**1. Grain Check (one row = one page per day):**
- Each row represents one page on one date
- Total rows in March 2026: 9,841,378
- Unique pages: ~427,149 (260,737 + 166,412)
- Verified: One row = one page per day

**2. Data Availability:**
- 6.8M rows (69%) have BOTH GSC and GA4 data
- 3.0M rows (31%) have ONLY GSC data
- Good data coverage for GSC signals

**3. AI Traffic Availability:**
- 6.8M rows have AI data columns available
- BUT only 5,534 rows (0.08%) actually have AI traffic
- AI traffic is VERY RARE - not useful for modeling

**4. What This Means for My Lane (Lane 1):**
- I'll focus on GSC data (available for ALL rows)
- GA4 data only for 69% of rows - I'll use it where available
- AI data is too rare to be useful for prediction

In [ ]:
# QUERY: Check which clients have which data sources
print("\n" + "-"*60)
print("QUERY: Data Source Availability")
print("-"*60)

query_sources = """
    SELECT
        client_has_gsc,
        client_has_ga4,
        COUNT(*) as row_count,
        COUNT(DISTINCT content_hash_id) as unique_pages
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
    WHERE month = '2026-03'
    GROUP BY client_has_gsc, client_has_ga4
"""
df_sources = con.execute(query_sources).df()
print(df_sources)

# QUERY: Check AI traffic availability
print("\n" + "-"*60)
print("QUERY: AI Traffic Availability")
print("-"*60)

query_ai = """
    SELECT
        COUNT(*) as total_rows,
        SUM(CASE WHEN sessions_ai IS NOT NULL THEN 1 ELSE 0 END) as has_ai_data,
        SUM(CASE WHEN sessions_ai > 0 THEN 1 ELSE 0 END) as has_ai_traffic
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
    WHERE month = '2026-03'
"""
df_ai = con.execute(query_ai).df()
print(df_ai)


------------------------------------------------------------
QUERY: Data Source Availability
------------------------------------------------------------


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   client_has_gsc  client_has_ga4  row_count  unique_pages
0            True            True    6822637        260737
1            True           False    3018741        166412

------------------------------------------------------------
QUERY: AI Traffic Availability
------------------------------------------------------------


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  has_ai_data  has_ai_traffic
0     9841378    6822637.0          5534.0


## 4. Data limits

### What this data CAN tell me:
1. **Which pages get impressions and clicks** (GSC data for ALL pages)
2. **Which pages have GA4 data** (6.8M rows - 69% of data)
3. **Which pages get AI traffic** (but only 5,534 rows out of 9.8M)
4. **Correlations** between signals like position and CTR

### What this data CANNOT tell me:

**1. Causation (Most Important!)**
- I can see that pages in better positions have higher CTR
- But I CANNOT say "better position causes higher CTR"
- Google's algorithm could be showing better pages higher AND people click them more

**2. Why users click**
- I see clicks and impressions
- But I don't know WHY someone clicked
- Is it the title? The snippet? The brand name?

**3. Content quality**
- I can measure performance (clicks, CTR)
- But I can't measure content quality directly
- A page might have low CTR but be high quality

**4. Why some pages have no GA4 data**
- 31% of rows (3M) have ONLY GSC data
- I don't know WHY these clients don't have GA4
- Could be technical issues, privacy settings, or they chose not to use it

**5. AI traffic is too rare to analyze**
- Only 5,534 rows have ANY AI traffic (0.08%)
- This is NOT enough to draw any conclusions
- I cannot say anything meaningful about AI traffic from this data

**6. New pages with no history**
- Pages with 0 impressions have no data to learn from
- My model only works for pages with some history

**7. External factors**
- I don't know about:
  - Seasonality (holidays, events)
  - Competitor actions
  - Google algorithm updates
  - Changes in search behavior

**8. Window overlaps (March vs June 2026)**
- What works in March might not work in June
- Search behavior changes over time
- I should test my findings before making decisions

### What this means for my analysis:

**I CAN:**
- ✅ Find signals associated with better performance
- ✅ Rank signals by importance
- ✅ Make directional recommendations
- ✅ Use GSC data confidently (available for ALL rows)

**I CANNOT:**
- ❌ Claim I "proved" what Google's algorithm does
- ❌ Say AI traffic matters (too rare to analyze)
- ❌ Make recommendations for brand new pages (no data)
- ❌ Use GA4 data for 31% of rows (only GSC available)
- ❌ Claim causation - only correlation

### My honest disclaimer:
> "All findings in this analysis are OBSERVATIONAL. I am looking at what happened, not why it happened. Any recommendations should be tested before being implemented."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.